In [158]:
from typing import List
import numpy as np
import stim
from stimcirq import stim_circuit_to_cirq_circuit, cirq_circuit_to_stim_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim, stim_pauli_string_to_cirq
from encoded.diagonalizing_circuit import get_measurement_circuit

In [142]:
generators = [
    stim.PauliString("ZIZII"),
    stim.PauliString("IZIZI"),
    stim.PauliString("IZZIZ")
]

In [143]:
for gi in generators:
    for gj in generators:
        assert gi.commutes(gj)

In [144]:
errors = [
    stim.PauliString("XIII"),
    stim.PauliString("IXII"),
    stim.PauliString("IIXI"),
    stim.PauliString("IIIX"),
]

In [145]:
number_true = 0
number_checked = 0
for i, ei in enumerate(errors):
    for j in range(i):
        number_checked += 1
        ej = errors[j]
        e = ei * ej
        commutators = []
        for generator in generators:
            comm = e.commutes(generator)
            commutators.append(comm)
        has_anticommuting_operator = any([not b for b in commutators])
        if has_anticommuting_operator:
            number_true += 1
        else:
            print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
print(f"{number_true}/{number_checked} operators anticommute.")

6/6 operators anticommute.


## Encoding circuit

In [164]:
tableau = stim.Tableau.from_stabilizers(generators, allow_underconstrained=True)
encoding_circuit = tableau.to_circuit()
print(encoding_circuit)

CX 1 0 2 0 3 0 4 0 1 2 3 1 3 2 4 2


In [166]:
for gen in generators:
    print(gen.before(encoding_circuit))

+Z____
+_Z___
+__Z__


In [170]:
encoding_circuit_cirq = stim_circuit_to_cirq_circuit(encoding_circuit)

## Logical fermionic operators

In [133]:
logical_ops_fermi = [
    of.FermionOperator("1^ 1"),
    of.FermionOperator("2^ 2"),
    of.FermionOperator("3^ 3"),
    of.FermionOperator("4^ 4"),
    of.FermionOperator("1^ 3"),
    of.FermionOperator("3^ 1"),
    of.FermionOperator("2^ 4"),
    of.FermionOperator("4^ 2"),
]

In [168]:
logical_ops_qubop = [of.transforms.jordan_wigner(lop) for lop in logical_ops_fermi]
logical_ops_cirq = [of.transforms.qubit_operator_to_pauli_sum(lop) for lop in logical_ops_qubop]

In [169]:
def conjugate_psum_with_circuit(psum: cirq.PauliSum, circuit: cirq.Circuit) -> cirq.PauliSum:
    new_pstrings = []
    for pstring in psum:
        new_pstrings.append(pstring.after(circuit))
    return cirq.PauliSum.from_pauli_strings(new_pstrings)

In [171]:
for lop in logical_ops_cirq:
    print(conjugate_psum_with_circuit(lop, encoding_circuit_cirq))

0.500*I-0.500*Z(q(1))*Z(q(3))
0.500*I-0.500*Z(q(1))*Z(q(2))*Z(q(4))
0.500*I-0.500*Z(q(3))
0.500*I-0.500*Z(q(4))
-0.250j*Z(q(2))*Y(q(3))*Z(q(4))-0.250*Z(q(2))*X(q(3))*Z(q(4))+0.250*Z(q(1))*Z(q(2))*X(q(3))*Z(q(4))+0.250j*Z(q(1))*Z(q(2))*Y(q(3))*Z(q(4))
0.250j*Z(q(2))*Y(q(3))*Z(q(4))+0.250*Z(q(1))*Z(q(2))*X(q(3))*Z(q(4))-0.250*Z(q(2))*X(q(3))*Z(q(4))-0.250j*Z(q(1))*Z(q(2))*Y(q(3))*Z(q(4))
-0.250j*Z(q(1))*Z(q(2))*Z(q(3))*Y(q(4))-0.250*Z(q(1))*Z(q(2))*Z(q(3))*X(q(4))+0.250*Z(q(3))*X(q(4))+0.250j*Z(q(3))*Y(q(4))
0.250j*Z(q(1))*Z(q(2))*Z(q(3))*Y(q(4))+0.250*Z(q(3))*X(q(4))-0.250*Z(q(1))*Z(q(2))*Z(q(3))*X(q(4))-0.250j*Z(q(3))*Y(q(4))
